# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Samad-17/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest, compared against Logistic Regression and a Decision Tree.

My lane (Refresh/Content Opportunity Scoring) is a ranking problem under limited reviewer capacity, so precision@K matters most — and Random Forest handles non-linear combinations of signals (position + CTR + volume interacting) better than a single linear model or shallow tree. This matches what the starter pipeline already showed; I'm validating it fresh here on real warehouse data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ["HF_TOKEN"]}');""")
BASE = "hf://datasets/FlyRank/internship-warehouse"
print("Setup done.")

Setup done.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Label (avoiding circularity): split each page's March 2026 data into an early window (days 1-15, for features) and a late window (days 16-31, for the label) — declining = late_clicks < early_clicks. This avoids reusing my Week-4 baseline rule's own score as the label, which would be circular.

Validation: client-holdout (grouped split) — clients kept fully in either train or test, never split across both, since pages from the same client can share patterns the model could otherwise memorize instead of generalizing.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
raw = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position,
           ga4_sessions, ga4_engaged_sessions
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
raw["report_date"] = pd.to_datetime(raw["report_date"])

early = raw[raw["report_date"].dt.day <= 15]
late = raw[raw["report_date"].dt.day > 15]

early_agg = early.groupby(["content_hash_id","client_hash_id"]).agg(
    impressions=("gsc_impressions","sum"), clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean"), sessions=("ga4_sessions","mean"),
    engaged_sessions=("ga4_engaged_sessions","mean")
).reset_index()

late_agg = late.groupby(["content_hash_id","client_hash_id"]).agg(late_clicks=("gsc_clicks","sum")).reset_index()

data = early_agg.merge(late_agg, on=["content_hash_id","client_hash_id"], how="inner")
data = data[data["impressions"] >= 50].copy()
data["label"] = (data["late_clicks"] < data["clicks"]).astype(int)
data["ctr"] = (data["clicks"] / data["impressions"]).fillna(0)
data["avg_position"] = data["avg_position"].fillna(100)
data["sessions"] = data["sessions"].fillna(0)
data["engaged_sessions"] = data["engaged_sessions"].fillna(0)

print("Rows:", len(data), "Declining rate:", round(data["label"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 92548 Declining rate: 0.289


In [3]:
rng = np.random.default_rng(42)
clients = data["client_hash_id"].unique()
test_clients = set(rng.choice(clients, size=max(1, int(len(clients)*0.2)), replace=False))

train = data[~data["client_hash_id"].isin(test_clients)].copy()
test = data[data["client_hash_id"].isin(test_clients)].copy()
print("Train rows:", len(train), "| Test rows:", len(test))
print("Train clients:", train["client_hash_id"].nunique(), "| Test clients:", test["client_hash_id"].nunique())

Train rows: 81476 | Test rows: 11072
Train clients: 32 | Test clients: 8


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = ["impressions","clicks","avg_position","ctr","sessions","engaged_sessions"]
X_train, y_train = train[features], train["label"]
X_test, y_test = test[features], test["label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
                                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))]),
    "decision_tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=30, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                                             class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:,1]
    results[name] = {
        "precision_at_20": round(precision_at_k(probs, y_test.values, 20), 3),
        "precision_at_50": round(precision_at_k(probs, y_test.values, 50), 3),
    }

# Baseline: same Week-4 rule, same test rows, same metric
test["baseline_score"] = test["avg_position"]*0.5 + (1 - test["ctr"])*50
results["baseline_rule"] = {
    "precision_at_20": round(precision_at_k(test["baseline_score"].values, y_test.values, 20), 3),
    "precision_at_50": round(precision_at_k(test["baseline_score"].values, y_test.values, 50), 3),
}

comparison = pd.DataFrame(results).T
comparison

,precision_at_20,precision_at_50
logistic_regression,0.75,0.84
decision_tree,0.70,0.64
random_forest,0.90,0.78
baseline_rule,0.00,0.06


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_model = models["random_forest"]
importances = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

ctr                 0.469627
clicks              0.423154
impressions         0.063919
avg_position        0.022591
sessions            0.017073
engaged_sessions    0.003636
dtype: float64


In [6]:
test["rf_prob"] = best_model.predict_proba(X_test)[:,1]
errors = test[(test["rf_prob"] > 0.5) != (test["label"] == 1)]
print("Error rows:", len(errors), "out of", len(test))
errors[["content_hash_id","impressions","ctr","avg_position","label","rf_prob"]].head(10)

Error rows: 2433 out of 11072


,content_hash_id,impressions,ctr,avg_position,label,rf_prob
181,content_0026e5dd76291725,586,0.003413,27.597151,0,0.801996
497,content_006889fde7c5736b,647,0.003091,4.248884,0,0.723776
650,content_00856c78a0d2c97b,3787,0.002377,8.310239,0,0.747664
673,content_00892819367d35e6,780,0.001282,4.970533,0,0.556038
732,content_009528b52c7d96ae,1838,0.001632,16.385675,0,0.721172
838,content_00a7afca11c5e9fb,795,0.003774,8.763435,0,0.766383
908,content_00b967cb01b7c40d,513,0.005848,2.140588,0,0.725861
915,content_00baca90889c56f7,662,0.003021,7.139091,0,0.765773
936,content_00bdebd065a8583b,498,0.002008,8.361421,0,0.698475
987,content_00c99a63642ef023,859,0.003492,8.572088,0,0.753192


Comparison table:

Model Precision@20	Precision@50

* logistic_regression	0.75	0.84

* decision_tree	0.70	0.64

* random_forest	0.90	0.78

* baseline_rule	0.00	0.06

The baseline rule performed very poorly here (0.00 precision@20) — essentially no better than random on this label. This makes sense: my Week-4 rule was built around visibility + weak position + low CTR as signs of an existing problem, but this week's label measures future click decline (late window vs. early window) — a different target than what the rule was designed to catch. All three models comfortably beat it, confirming a learned model captures this forward-looking pattern far better than a static current-state rule.

Interestingly, random forest wins at precision@20 (0.90) but logistic regression edges ahead at precision@50 (0.84 vs 0.78) — a useful, honest nuance: no single model dominates at every K, and reporting only one cutoff would have overstated random forest's advantage.

Feature importance (random forest): ctr (0.470) and clicks (0.423) dominate, together accounting for ~89% of the model's decisions — impressions, avg_position, sessions, and engaged_sessions contribute comparatively little.

Important caveat on this result: because my label is defined as late_clicks < early_clicks, pages with high early clicks are mechanically more likely to see a relative drop (regression toward the mean) — so the model leaning heavily on clicks/ctr may partly reflect this construction, not purely a causal "low CTR predicts decline" signal. This isn't classic leakage (no future data enters the features), but it is a labeling artifact worth flagging: a stronger design in a later pass would use a longer, non-adjacent prior window (e.g., prior month → next month) rather than splitting one month in half.

Error inspection: the sample errors above are all false positives — pages the model scored as high-risk (rf_prob 0.55–0.80) that were actually labeled "not declining" (label=0). All ten share a pattern: very low CTR (0.001–0.006) despite meaningful impression volume (500–3,800) and weak average position (2–28). The model is reacting to a genuinely weak-looking page, but in these specific cases clicks didn't drop further in the late window — likely because these pages were already so low-performing that there wasn't much room left to decline. This suggests the model conflates "chronically weak" with "actively declining," which are related but not identical — a useful distinction to sharpen in a future label design.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.